# 🎓 SyLOC-T : Projet BDA & Développement Web (CROUS de Thiès)
## 📊 Notebook d'Analyse BDA & Déploiement Cloud avec Ngrok / Colab

Ce notebook permet de :
1. **Cloner le projet** et installer l'environnement.
2. **Initialiser la base de données** (mode SQLite optimisé pour le Cloud).
3. **Exécuter les requêtes SQL BDA avancées** et visualiser les données avec `pandas` / `seaborn`.
4. **Exposer l'application en direct avec Ngrok** (lien HTTPS sécurisé pour la démo).

### 1️⃣ Étape 1 : Cloner le dépôt et installer les dépendances

In [ ]:
import os
from getpass import getpass

# Si le dépôt est privé, collez votre Personal Access Token GitHub (ghp_...)
# Si le dépôt est public, appuyez simplement sur Entrée
token = getpass("🔑 GitHub Token (laisser vide si repo public) : ").strip()

!rm -rf SyLOC-T
if token:
    !git clone -b develop https://{token}@github.com/mhdlamine21/SyLOC-T.git
else:
    !git clone -b develop https://github.com/mhdlamine21/SyLOC-T.git

%cd SyLOC-T

# Installation des dépendances Backend, BDA et Ngrok
!pip install -r vcn_backend/requirements.txt
!pip install pandas matplotlib seaborn pyngrok

### 2️⃣ Étape 2 : Initialiser la Base de Données (Mode Cloud SQLite)

In [ ]:
import os

# 1. Configuration automatique du mode SQLite pour Google Colab
os.environ["DB_ENGINE"] = "sqlite"
with open("vcn_backend/.env", "w") as f:
    f.write("SECRET_KEY=django-insecure-colab-key-syloc\n")
    f.write("DEBUG=True\n")
    f.write("DB_ENGINE=sqlite\n")
    f.write("ALLOWED_HOSTS=*\n")
    f.write("CORS_ALLOWED_ORIGINS=http://localhost:5173,http://localhost:3000\n")

# 2. Application des migrations et injection des données sénégalaises CROUS
%cd vcn_backend
!python manage.py migrate
!python seed.py
%cd ..

### 3️⃣ Étape 3 : Requêtes SQL Avancées & Graphiques BDA

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Connexion à la BDD SQLite du projet
conn = sqlite3.connect('vcn_backend/db.sqlite3')

# --------------------------------------------------------
# 🏢 Requête 1 : Répartition du patrimoine CROUS-T & surfaces
# --------------------------------------------------------
query_locaux = """
SELECT 
    type_local AS Type,
    CASE WHEN est_libre = 1 THEN 'Disponible' ELSE 'Occupe' END AS Disponibilite,
    etat_physique AS Etat,
    COUNT(*) AS Total,
    ROUND(AVG(surface_m2), 1) AS Surface_Moy_m2
FROM patrimoine_local
GROUP BY type_local, est_libre, etat_physique
ORDER BY Total DESC;
"""
df_locaux = pd.read_sql_query(query_locaux, conn)
print("=== 🏢 1. ANALYSE DU PARC IMMOBILIER CROUS-T ===")
display(df_locaux)

# Graphique 1
plt.figure(figsize=(10, 5))
sns.set_theme(style="whitegrid")
sns.barplot(data=df_locaux, x='Type', y='Total', hue='Disponibilite', palette='Set2')
plt.title('Répartition des Locaux par Type et Disponibilité (SyLOC-T)')
plt.ylabel('Nombre de Locaux')
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

# --------------------------------------------------------
# 💳 Requête 2 : Analyse des Encaissements (Wave / OM / Espèces)
# --------------------------------------------------------
query_paiements = """
SELECT 
    mode AS Mode_Paiement,
    statut AS Statut_Paiement,
    COUNT(*) AS Nombre_Transactions,
    ROUND(SUM(montant_regle), 0) AS Total_Encaisse_FCFA
FROM paiements_paiement
GROUP BY mode, statut;
"""
df_paiements = pd.read_sql_query(query_paiements, conn)
print("\n=== 💳 2. ANALYSE FINANCIERE DES PAIEMENTS ===")
display(df_paiements)

# Graphique 2 (Camembert des modes de paiement)
if not df_paiements.empty and df_paiements['Total_Encaisse_FCFA'].sum() > 0:
    plt.figure(figsize=(6, 6))
    plt.pie(df_paiements['Total_Encaisse_FCFA'], labels=df_paiements['Mode_Paiement'], autopct='%1.1f%%', colors=['#3498db','#e74c3c','#2ecc71'])
    plt.title('Répartition des Encaissements par Mode de Paiement')
    plt.show()

### 4️⃣ Étape 4 : Déploiement avec Ngrok (Accès Public HTTPS)

In [ ]:
import subprocess
import time
from pyngrok import ngrok
from getpass import getpass

# 1. Saisie de votre authtoken gratuit Ngrok (obtenu sur dashboard.ngrok.com)
ngrok_token = getpass("🔑 Entrez votre Authtoken Ngrok : ").strip()
ngrok.set_auth_token(ngrok_token)

# 2. Démarrage du serveur Django en arrière-plan
print("🚀 Lancement du serveur Django...")
backend_proc = subprocess.Popen(["python", "vcn_backend/manage.py", "runserver", "0.0.0.0:8000"])
time.sleep(3)

# 3. Ouverture du tunnel sécurisé Ngrok vers le port 8000
tunnel = ngrok.connect(8000)
public_url = tunnel.public_url

print("\n" + "="*60)
print(f"🎉 Application SyLOC-T en ligne via Ngrok !")
print(f"🌐 URL de l'API       : {public_url}")
print(f"📚 Swagger Docs       : {public_url}/api/docs/")
print(f"👥 Données Publiques  : {public_url}/api/public/stats/")
print("="*60)